In [1]:
!pip install -q librosa timm

import os, glob, random
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

cuda


In [2]:
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
g2i = {g: i for i, g in enumerate(GENRES)}
STEMS = ['drums', 'vocals', 'bass', 'other']
BASE = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'

SR, DUR, N_MELS = 22050, 10, 128

songs = {g: [] for g in GENRES}
for g in GENRES:
    gd = os.path.join(BASE, 'genres_stems', g)
    if os.path.exists(gd):
        for s in os.listdir(gd):
            sp = os.path.join(gd, s)
            if os.path.isdir(sp) and all(os.path.exists(os.path.join(sp, f"{st}.wav")) for st in STEMS):
                songs[g].append(sp)

noise = glob.glob(os.path.join(BASE, 'ESC-50-master', 'audio', '*.wav'))
print(f"Songs: {sum(len(v) for v in songs.values())}, Noise: {len(noise)}")

Songs: 1000, Noise: 2000


In [3]:
def load(p, sr=SR, d=DUR):
    try:
        a, _ = librosa.load(p, sr=sr, duration=d)
        t = sr * d
        if len(a) < t: a = np.tile(a, 3)[:t]
        return a[:t]
    except: return np.zeros(sr * d)

def mix_cross(g):
    m = np.zeros(SR * DUR, dtype=np.float32)
    for st in STEMS:
        m += load(os.path.join(random.choice(songs[g]), f"{st}.wav"))
    return m

def add_noise(a, lvl):
    n = load(random.choice(noise)) if noise else np.zeros_like(a)
    if np.max(np.abs(n)) > 0: n /= np.max(np.abs(n))
    return a + lvl * n

def norm(a):
    a = a - np.mean(a)
    if np.max(np.abs(a)) > 0: a = a / np.max(np.abs(a)) * 0.95
    return a.astype(np.float32)

def to_mel(a):
    m = librosa.feature.melspectrogram(y=a, sr=SR, n_mels=N_MELS, n_fft=2048, hop_length=512)
    m = librosa.power_to_db(m, ref=np.max)
    return (m - m.mean()) / (m.std() + 1e-6)

In [4]:
class DS(Dataset):
    def __init__(self, n=250):  # MORE SAMPLES
        self.d = [(g, i) for g in GENRES for i in range(n)]
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        g, _ = self.d[i]
        a = mix_cross(g)
        
        # HIGHER NOISE: 80% prob, level 0.15-0.45 (was 0.1-0.35)
        if random.random() < 0.80:
            a = add_noise(a, random.uniform(0.15, 0.45))
        
        if random.random() < 0.5:
            a = np.roll(a, random.randint(-SR//2, SR//2))
        
        a = norm(a)
        m = to_mel(a)
        
        # SpecAugment
        if random.random() < 0.5:
            t = random.randint(0, 25)
            t0 = random.randint(0, max(1, m.shape[1]-t-1))
            m[:, t0:t0+t] = 0
        if random.random() < 0.5:
            f = random.randint(0, 15)
            f0 = random.randint(0, max(1, m.shape[0]-f-1))
            m[f0:f0+f, :] = 0
        
        return torch.tensor(m, dtype=torch.float32).unsqueeze(0).repeat(3,1,1), g2i[g]

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.bb = timm.create_model('efficientnet_b2', pretrained=True, num_classes=0)
        self.h = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(self.bb.num_features, 256),
            nn.ReLU(), nn.Dropout(0.25), nn.Linear(256, 10)
        )
    def forward(self, x): return self.h(self.bb(x))

In [5]:
def train_model(seed, epochs=12):
    print(f"\n{'='*40}")
    print(f"TRAINING SEED {seed}")
    print(f"{'='*40}")
    
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    model = Model().to(device)
    tr = DataLoader(DS(250), batch_size=24, shuffle=True, num_workers=2)  # 250 samples
    
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    opt = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=0.01)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=8e-4, epochs=epochs, steps_per_epoch=len(tr))
    
    best = 0
    for ep in range(epochs):
        model.train()
        c, t = 0, 0
        for d, l in tqdm(tr, desc=f"S{seed} Ep{ep+1}"):
            d, l = d.to(device), l.to(device)
            opt.zero_grad()
            o = model(d)
            crit(o, l).backward()
            opt.step()
            sch.step()
            c += (o.argmax(1) == l).sum().item()
            t += l.size(0)
        acc = c / t
        if acc > best:
            best = acc
            torch.save(model.state_dict(), f'model_{seed}.pth')
        print(f"Acc: {acc:.4f}, Best: {best:.4f}")
    
    model.load_state_dict(torch.load(f'model_{seed}.pth'))
    return model

In [6]:
# 5 MODELS - more than v7's 3
models = []
for seed in [42, 123, 777, 999, 2024]:
    m = train_model(seed, epochs=12)
    models.append(m)
print(f"\nTrained {len(models)} models!")


TRAINING SEED 42


model.safetensors:   0%|          | 0.00/36.8M [00:00<?, ?B/s]

S42 Ep1: 100%|██████████| 105/105 [05:02<00:00,  2.88s/it]


Acc: 0.2476, Best: 0.2476


S42 Ep2: 100%|██████████| 105/105 [04:17<00:00,  2.45s/it]


...

Acc: 0.9200, Best: 0.9200


S2024 Ep11: 100%|██████████| 105/105 [04:13<00:00,  2.41s/it]


Acc: 0.9368, Best: 0.9368


S2024 Ep12: 100%|██████████| 105/105 [04:14<00:00,  2.42s/it]


Acc: 0.9324, Best: 0.9368

Trained 5 models!


In [7]:
# Inference
for m in models:
    m.eval()

test_df = pd.read_csv(os.path.join(BASE, 'test.csv'))
sub = pd.read_csv(os.path.join(BASE, 'sample_submission.csv'))
i2f = dict(zip(test_df['id'], test_df['filename']))
files = [os.path.join(BASE, i2f[r['id']]) for _, r in sub.iterrows()]

def ensemble_tta(path, n_crops=5):
    try:
        af, _ = librosa.load(path, sr=SR)
    except:
        af = np.zeros(SR * 20)
    
    t = SR * DUR
    if len(af) < t: af = np.tile(af, 3)
    
    all_probs = []
    
    for model in models:
        for p in np.linspace(0, max(0, len(af)-t), n_crops).astype(int):
            cr = af[p:p+t]
            if len(cr) < t: cr = np.pad(cr, (0, t-len(cr)))
            cr = norm(cr)
            m = to_mel(cr)
            mt = torch.tensor(m, dtype=torch.float32).unsqueeze(0).unsqueeze(0).repeat(1,3,1,1).to(device)
            with torch.no_grad():
                all_probs.append(torch.softmax(model(mt), dim=1))
    
    # 5 models x 5 crops = 25 predictions
    return torch.stack(all_probs).mean(0).argmax(1).item()

print(f"Ensemble (5 models x 5 crops)...")
preds = [ensemble_tta(f) for f in tqdm(files)]

sub['genre'] = [GENRES[p] for p in preds]
sub.to_csv('submission.csv', index=False)
print("Done!")
print(sub['genre'].value_counts())

Ensemble (5 models x 5 crops)...


100%|██████████| 3020/3020 [47:24<00:00,  1.06it/s]

Done!
genre
reggae       320
disco        319
pop          317
metal        311
rock         307
blues        298
classical    296
hiphop       294
jazz         288
country      270
Name: count, dtype: int64
